In [0]:
# Bank Marketing Campaign Analysis
# Dataset: Portuguese Banking Institution Marketing Campaign

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("📦 Libraries loaded successfully!")
print("\n" + "="*70)
print("LOADING DATASET")
print("="*70)

# Load the full bank dataset
df = pd.read_csv('bank-full.csv', delimiter=';')

print(f"\n✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [0]:
# Display first few rows
print("="*70)
print("FIRST 5 ROWS OF DATA")
print("="*70)
display(df.head())

print("\n" + "="*70)
print("LAST 5 ROWS OF DATA")
print("="*70)
display(df.tail())

print("\n" + "="*70)
print("RANDOM SAMPLE (10 rows)")
print("="*70)
display(df.sample(10, random_state=42))

In [0]:
# Examine data structure
print("="*70)
print("COLUMN INFORMATION")
print("="*70)
print(f"\nTotal columns: {len(df.columns)}\n")

print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "="*70)
print("DATA TYPES")
print("="*70)
print(df.dtypes)

print("\n" + "="*70)
print("DETAILED INFO")
print("="*70)
df.info()

In [0]:
# Check for missing values
print("="*70)
print("MISSING VALUES ANALYSIS")
print("="*70)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("\n⚠️  Columns with missing values:\n")
    display(missing_df)
else:
    print("\n✅ No missing values detected in any column!")
    print("\nThis is excellent for analysis - no imputation needed.")

print(f"\n📊 Total missing values: {df.isnull().sum().sum()}")
print(f"📊 Data completeness: {((1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100):.2f}%")

In [0]:
# Descriptive statistics for numerical columns
print("="*70)
print("NUMERICAL FEATURES - DESCRIPTIVE STATISTICS")
print("="*70)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}\n")

display(df[numerical_cols].describe().T)

# Additional statistics
print("\n" + "="*70)
print("ADDITIONAL STATISTICS")
print("="*70)

for col in numerical_cols:
    print(f"\n{col}:")
    print(f"  Median: {df[col].median():.2f}")
    print(f"  Mode: {df[col].mode().values[0] if len(df[col].mode()) > 0 else 'N/A'}")
    print(f"  Skewness: {df[col].skew():.2f}")
    print(f"  Kurtosis: {df[col].kurtosis():.2f}")

In [0]:
# Descriptive statistics for categorical columns
print("="*70)
print("CATEGORICAL FEATURES - UNIQUE VALUES")
print("="*70)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}\n")

for col in categorical_cols:
    n_unique = df[col].nunique()
    print(f"\n{col}:")
    print(f"  Unique values: {n_unique}")
    print(f"  Value counts:")
    print(df[col].value_counts().to_string())
    print()

In [0]:
# Analyze target variable (likely 'y' - whether client subscribed)
print("="*70)
print("TARGET VARIABLE ANALYSIS")
print("="*70)

# Assuming 'y' is the target variable
if 'y' in df.columns:
    target = 'y'
    print(f"\nTarget variable: '{target}'")
    print(f"\nValue counts:")
    print(df[target].value_counts())
    
    print(f"\nProportions:")
    print(df[target].value_counts(normalize=True))
    
    # Visualize target distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    df[target].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
    axes[0].set_title('Target Variable Distribution (Count)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Subscribed Term Deposit?', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(df[target].value_counts().values):
        axes[0].text(i, v + 500, str(v), ha='center', fontsize=11, fontweight='bold')
    
    # Pie chart
    df[target].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                    colors=['steelblue', 'coral'], startangle=90)
    axes[1].set_title('Target Variable Distribution (%)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    # Check for class imbalance
    imbalance_ratio = df[target].value_counts().iloc[0] / df[target].value_counts().iloc[1]
    print(f"\n📊 Class imbalance ratio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio > 3:
        print("\n⚠️  SEVERE CLASS IMBALANCE DETECTED!")
        print("    Consider using techniques like:")
        print("    - SMOTE (Synthetic Minority Over-sampling)")
        print("    - Class weights in model training")
        print("    - Threshold optimization")
        print("    - Ensemble methods")
    elif imbalance_ratio > 1.5:
        print("\n⚠️  Moderate class imbalance detected")
        print("    Consider using class_weight='balanced' in models")
    else:
        print("\n✅ Classes are relatively balanced")
else:
    print("\n⚠️  Target variable 'y' not found in dataset")

In [0]:
# Visualize distributions of numerical features
print("="*70)
print("NUMERICAL FEATURES - DISTRIBUTIONS")
print("="*70)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Create histograms
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:6]):  # Plot first 6 numerical columns
    if idx < len(axes):
        df[col].hist(bins=30, ax=axes[idx], color='steelblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel(col, fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].grid(axis='y', alpha=0.3)
        
        # Add mean and median lines
        mean_val = df[col].mean()
        median_val = df[col].median()
        axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
        axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
        axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n📈 Observations:")
for col in numerical_cols:
    skew_val = df[col].skew()
    if abs(skew_val) > 1:
        print(f"  • {col}: Highly skewed (skewness = {skew_val:.2f})")
    elif abs(skew_val) > 0.5:
        print(f"  • {col}: Moderately skewed (skewness = {skew_val:.2f})")
    else:
        print(f"  • {col}: Approximately normal (skewness = {skew_val:.2f})")

In [0]:
# Correlation analysis
print("="*70)
print("CORRELATION ANALYSIS")
print("="*70)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 1:
    # Compute correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    print("\nCorrelation Matrix:\n")
    display(correlation_matrix.round(3))
    
    # Visualize correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated pairs
    print("\n" + "="*70)
    print("HIGHLY CORRELATED FEATURE PAIRS (|correlation| > 0.7)")
    print("="*70)
    
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > 0.7:
                high_corr_pairs.append({
                    'Feature 1': correlation_matrix.columns[i],
                    'Feature 2': correlation_matrix.columns[j],
                    'Correlation': correlation_matrix.iloc[i, j]
                })
    
    if high_corr_pairs:
        high_corr_df = pd.DataFrame(high_corr_pairs)
        display(high_corr_df)
        print("\n⚠️  Consider removing one feature from each highly correlated pair to reduce multicollinearity")
    else:
        print("\n✅ No highly correlated feature pairs found")
else:
    print("\n⚠️  Not enough numerical columns for correlation analysis")

In [0]:
# 📊 EDA SUMMARY - KEY FINDINGS

print("="*70)
print("EXPLORATORY DATA ANALYSIS - SUMMARY")
print("="*70)

print("\n📁 DATASET OVERVIEW:")
print(f"  • Total samples: {df.shape[0]:,}")
print(f"  • Total features: {df.shape[1]} (7 numerical, 10 categorical)")
print(f"  • Missing values: 0 (100% complete data ✅)")
print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n🎯 TARGET VARIABLE (y):")
print(f"  • No (did not subscribe): {(df['y']=='no').sum():,} ({(df['y']=='no').sum()/len(df)*100:.1f}%)")
print(f"  • Yes (subscribed): {(df['y']=='yes').sum():,} ({(df['y']=='yes').sum()/len(df)*100:.1f}%)")
print(f"  • Class imbalance ratio: 7.55:1 ⚠️ SEVERE")
print("  • This is a binary classification problem with high imbalance")

print("\n📊 NUMERICAL FEATURES:")
print("  • age: Mean=40.9 years, mostly 33-48 range")
print("  • balance: Highly skewed (mean=$1,362, median=$448)")
print("  • duration: Call duration in seconds (highly predictive but NOT usable before campaign)")
print("  • campaign: Number of contacts this campaign (1-3 typical)")
print("  • pdays: Days since last contact (-1 = never contacted)")
print("  • previous: Previous campaign contacts (most = 0)")
print("  • day: Day of month (uniformly distributed)")

print("\n🏷️ CATEGORICAL FEATURES:")
print("  • job: 12 types, dominated by blue-collar (21.5%) and management (20.9%)")
print("  • marital: married (60.2%), single (28.3%), divorced (11.5%)")
print("  • education: secondary (51.3%), tertiary (29.4%), primary (15.1%)")
print("  • default: credit default rare (1.8% yes)")
print("  • housing: has housing loan (55.6% yes)")
print("  • loan: has personal loan (16.0% yes)")
print("  • contact: cellular (64.8%), unknown (28.8%), telephone (6.4%)")
print("  • month: peaked in May (30.4%), Jul, Aug, Jun")
print("  • poutcome: previous campaign outcome - mostly unknown (81.7%)")

print("\n🔗 CORRELATIONS:")
print("  • No highly correlated features (all |r| < 0.7) ✅")
print("  • Strongest correlation: pdays ↔ previous (r=0.455)")
print("  • Features are relatively independent - low multicollinearity")

print("\n📊 DATA DISTRIBUTION ISSUES:")
print("  • Highly skewed: balance, duration, campaign, pdays, previous")
print("  • Consider log transformation or binning for modeling")
print("  • Outliers present in balance (max=$102,127 vs median=$448)")

print("\n⚠️  IMPORTANT NOTES:")
print("  • 'duration' is highly predictive BUT cannot be used for prediction")
print("    (call duration is only known AFTER the call ends)")
print("  • For real-world deployment, must exclude 'duration' from features")
print("  • Class imbalance requires special handling (SMOTE, class weights, etc.)")

print("\n" + "="*70)
print("🚀 RECOMMENDED NEXT STEPS")
print("="*70)
print("\n1. FEATURE ENGINEERING:")
print("   - Create age groups (young, middle-aged, senior)")
print("   - Bin balance into categories (negative, low, medium, high)")
print("   - Create 'contacted_before' binary from pdays")
print("   - Encode categorical variables (one-hot or label encoding)")
print("   - Handle skewed distributions (log transform)")

print("\n2. DATA PREPROCESSING:")
print("   - Scale numerical features (StandardScaler or MinMaxScaler)")
print("   - Handle class imbalance:")
print("     • Use class_weight='balanced' in models")
print("     • Try SMOTE for oversampling minority class")
print("     • Optimize threshold (like we did in W2D2 notebook!)")

print("\n3. MODELING STRATEGY:")
print("   - Baseline: Logistic Regression with class weights")
print("   - Tree-based: Random Forest, XGBoost, LightGBM")
print("   - Ensemble methods handle imbalance better")
print("   - Remember: EXCLUDE 'duration' for realistic predictions")

print("\n4. EVALUATION METRICS:")
print("   - DON'T use accuracy (misleading with imbalanced data!)")
print("   - PRIMARY: Recall (catch subscribers), Precision (avoid false alarms)")
print("   - SECONDARY: F1-score, ROC AUC, PR AUC")
print("   - Consider business costs of FP vs FN")

print("\n5. MODEL INTERPRETABILITY:")
print("   - Feature importance analysis")
print("   - SHAP values for model explanations")
print("   - Identify key factors driving subscriptions")

print("\n" + "="*70)
print("✅ EDA COMPLETE - Ready for Feature Engineering & Modeling!")
print("="*70)

In [0]:
# 📊 KEY PREDICTIVE PATTERNS - ORIGINAL FEATURES VS TARGET
# Visualizing what drives customer subscription decisions

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

print("="*70)
print("ANALYZING SUBSCRIPTION DRIVERS - ORIGINAL FEATURES ONLY")
print("="*70)

# Set style
sns.set_style('whitegrid')
sns.set_palette('Set2')

# ============================================================================
# FIGURE 1: AGE DISTRIBUTION BY SUBSCRIPTION STATUS
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
ax1 = axes[0]
for status in ['no', 'yes']:
    data = df[df['y'] == status]['age']
    ax1.hist(data, bins=30, alpha=0.6, label=f"{'Subscribed' if status == 'yes' else 'Not Subscribed'}", edgecolor='black')
ax1.set_xlabel('Age', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count', fontsize=12, fontweight='bold')
ax1.set_title('Age Distribution by Subscription Status', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# Age boxplot
ax2 = axes[1]
sns.boxplot(data=df, x='y', y='age', ax=ax2, palette=['steelblue', 'coral'])
ax2.set_xlabel('Subscribed?', fontsize=12, fontweight='bold')
ax2.set_ylabel('Age', fontsize=12, fontweight='bold')
ax2.set_title('Age Comparison: Subscribers vs Non-Subscribers', fontsize=13, fontweight='bold')
ax2.set_xticklabels(['No', 'Yes'])

# Subscription rate by age group
ax3 = axes[2]
age_bins = [0, 30, 40, 50, 60, 100]
age_labels = ['<30', '30-39', '40-49', '50-59', '60+']
df_temp = df.copy()
df_temp['age_bin'] = pd.cut(df_temp['age'], bins=age_bins, labels=age_labels)
subscription_by_age = df_temp.groupby('age_bin')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_age = plt.cm.viridis(np.linspace(0.3, 0.9, len(subscription_by_age)))
subscription_by_age.plot(kind='bar', ax=ax3, color=colors_age, edgecolor='black')
ax3.set_xlabel('Age Group', fontsize=12, fontweight='bold')
ax3.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax3.set_title('Subscription Rate by Age Group', fontsize=13, fontweight='bold')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.grid(axis='y', alpha=0.3)
for i, v in enumerate(subscription_by_age):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📈 AGE INSIGHTS:")
print(f"  • Average age (subscribers): {df[df['y']=='yes']['age'].mean():.1f} years")
print(f"  • Average age (non-subscribers): {df[df['y']=='no']['age'].mean():.1f} years")
print(f"  • Highest subscription rate: {subscription_by_age.idxmax()} age group ({subscription_by_age.max():.1f}%)")

# ============================================================================
# FIGURE 2: ACCOUNT BALANCE IMPACT
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Balance distribution (log scale for better visibility)
ax1 = axes[0]
for status in ['no', 'yes']:
    data = df[df['y'] == status]['balance']
    # Filter extreme outliers for visualization
    data_filtered = data[(data > -2000) & (data < 10000)]
    ax1.hist(data_filtered, bins=50, alpha=0.6, label=f"{'Subscribed' if status == 'yes' else 'Not Subscribed'}", edgecolor='black')
ax1.set_xlabel('Account Balance ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count', fontsize=12, fontweight='bold')
ax1.set_title('Balance Distribution by Subscription Status\n(Filtered: -$2K to $10K)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# Balance boxplot
ax2 = axes[1]
# Filter outliers for better visualization
df_filtered = df[(df['balance'] > -2000) & (df['balance'] < 10000)]
sns.boxplot(data=df_filtered, x='y', y='balance', ax=ax2, palette=['steelblue', 'coral'])
ax2.set_xlabel('Subscribed?', fontsize=12, fontweight='bold')
ax2.set_ylabel('Balance ($)', fontsize=12, fontweight='bold')
ax2.set_title('Balance Comparison\n(Filtered: -$2K to $10K)', fontsize=13, fontweight='bold')
ax2.set_xticklabels(['No', 'Yes'])

# Subscription rate by balance category
ax3 = axes[2]
balance_bins = [-np.inf, 0, 500, 2000, np.inf]
balance_labels = ['Negative', 'Low\n($0-$500)', 'Medium\n($500-$2K)', 'High\n(>$2K)']
df_temp = df.copy()
df_temp['balance_bin'] = pd.cut(df_temp['balance'], bins=balance_bins, labels=balance_labels)
subscription_by_balance = df_temp.groupby('balance_bin')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_bal = plt.cm.plasma(np.linspace(0.2, 0.8, len(subscription_by_balance)))
subscription_by_balance.plot(kind='bar', ax=ax3, color=colors_bal, edgecolor='black')
ax3.set_xlabel('Balance Category', fontsize=12, fontweight='bold')
ax3.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax3.set_title('Subscription Rate by Balance Category', fontsize=13, fontweight='bold')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.grid(axis='y', alpha=0.3)
for i, v in enumerate(subscription_by_balance):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💰 BALANCE INSIGHTS:")
print(f"  • Median balance (subscribers): ${df[df['y']=='yes']['balance'].median():,.0f}")
print(f"  • Median balance (non-subscribers): ${df[df['y']=='no']['balance'].median():,.0f}")
print(f"  • Highest subscription rate: {subscription_by_balance.idxmax()} ({subscription_by_balance.max():.1f}%)")

# ============================================================================
# FIGURE 3: CATEGORICAL FEATURES - JOB, MARITAL, EDUCATION
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Job type subscription rates
ax1 = axes[0]
job_subscription = df.groupby('job')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100).sort_values(ascending=False)
colors_job = plt.cm.tab20(np.linspace(0, 1, len(job_subscription)))
job_subscription.plot(kind='barh', ax=ax1, color=colors_job, edgecolor='black')
ax1.set_xlabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Job Type', fontsize=12, fontweight='bold')
ax1.set_title('Subscription Rate by Job Type', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(job_subscription):
    ax1.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9, fontweight='bold')

# Marital status
ax2 = axes[1]
marital_subscription = df.groupby('marital')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100).sort_values(ascending=False)
colors_marital = ['#ff6b6b', '#4ecdc4', '#45b7d1']
marital_subscription.plot(kind='bar', ax=ax2, color=colors_marital, edgecolor='black')
ax2.set_xlabel('Marital Status', fontsize=12, fontweight='bold')
ax2.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax2.set_title('Subscription Rate by Marital Status', fontsize=13, fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.grid(axis='y', alpha=0.3)
for i, v in enumerate(marital_subscription):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Education level
ax3 = axes[2]
edu_subscription = df.groupby('education')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100).sort_values(ascending=False)
colors_edu = ['#95e1d3', '#f38181', '#fce38a', '#c5cae9']
edu_subscription.plot(kind='bar', ax=ax3, color=colors_edu, edgecolor='black')
ax3.set_xlabel('Education Level', fontsize=12, fontweight='bold')
ax3.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax3.set_title('Subscription Rate by Education', fontsize=13, fontweight='bold')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.grid(axis='y', alpha=0.3)
for i, v in enumerate(edu_subscription):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n👔 DEMOGRAPHIC INSIGHTS:")
print(f"  • Best job for subscription: {job_subscription.idxmax()} ({job_subscription.max():.1f}%)")
print(f"  • Best marital status: {marital_subscription.idxmax()} ({marital_subscription.max():.1f}%)")
print(f"  • Best education level: {edu_subscription.idxmax()} ({edu_subscription.max():.1f}%)")

# ============================================================================
# FIGURE 4: CONTACT METHOD & CAMPAIGN TIMING
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Contact method
ax1 = axes[0, 0]
contact_subscription = df.groupby('contact')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100).sort_values(ascending=False)
colors_contact = ['#2ecc71', '#e74c3c', '#f39c12']
contact_subscription.plot(kind='bar', ax=ax1, color=colors_contact, edgecolor='black')
ax1.set_xlabel('Contact Method', fontsize=12, fontweight='bold')
ax1.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax1.set_title('Subscription Rate by Contact Method', fontsize=13, fontweight='bold')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(contact_subscription):
    ax1.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Month of contact
ax2 = axes[0, 1]
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_subscription = df.groupby('month')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
month_subscription = month_subscription.reindex(month_order)
colors_month = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(month_subscription)))
month_subscription.plot(kind='bar', ax=ax2, color=colors_month, edgecolor='black')
ax2.set_xlabel('Month of Contact', fontsize=12, fontweight='bold')
ax2.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax2.set_title('Subscription Rate by Month', fontsize=13, fontweight='bold')
ax2.set_xticklabels([m.upper() for m in month_order], rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)
for i, v in enumerate(month_subscription):
    if not np.isnan(v):
        ax2.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Campaign contacts
ax3 = axes[1, 0]
campaign_bins = [0, 1, 2, 3, 5, 100]
campaign_labels = ['1', '2', '3', '4-5', '6+']
df_temp = df.copy()
df_temp['campaign_bin'] = pd.cut(df_temp['campaign'], bins=campaign_bins, labels=campaign_labels)
campaign_subscription = df_temp.groupby('campaign_bin')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_camp = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(campaign_subscription)))
campaign_subscription.plot(kind='bar', ax=ax3, color=colors_camp, edgecolor='black')
ax3.set_xlabel('Number of Contacts This Campaign', fontsize=12, fontweight='bold')
ax3.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax3.set_title('Subscription Rate by Campaign Intensity', fontsize=13, fontweight='bold')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.grid(axis='y', alpha=0.3)
for i, v in enumerate(campaign_subscription):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Previous campaign outcome
ax4 = axes[1, 1]
poutcome_subscription = df.groupby('poutcome')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100).sort_values(ascending=False)
colors_pout = ['#e74c3c', '#2ecc71', '#f39c12', '#3498db']
poutcome_subscription.plot(kind='bar', ax=ax4, color=colors_pout, edgecolor='black')
ax4.set_xlabel('Previous Campaign Outcome', fontsize=12, fontweight='bold')
ax4.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax4.set_title('Subscription Rate by Previous Outcome', fontsize=13, fontweight='bold')
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
ax4.grid(axis='y', alpha=0.3)
for i, v in enumerate(poutcome_subscription):
    ax4.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📞 CONTACT & TIMING INSIGHTS:")
print(f"  • Best contact method: {contact_subscription.idxmax()} ({contact_subscription.max():.1f}%)")
print(f"  • Best month: {month_subscription.idxmax().upper()} ({month_subscription.max():.1f}%)")
print(f"  • Campaign intensity impact: Fewer contacts = higher success")
print(f"  • Previous success matters: {poutcome_subscription.loc['success']:.1f}% subscription rate")

# ============================================================================
# FIGURE 5: FINANCIAL PRODUCTS (HOUSING LOAN, PERSONAL LOAN, DEFAULT)
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Housing loan
ax1 = axes[0]
housing_subscription = df.groupby('housing')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_housing = ['#3498db', '#e74c3c']
housing_subscription.plot(kind='bar', ax=ax1, color=colors_housing, edgecolor='black')
ax1.set_xlabel('Has Housing Loan?', fontsize=12, fontweight='bold')
ax1.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax1.set_title('Subscription Rate by Housing Loan Status', fontsize=13, fontweight='bold')
ax1.set_xticklabels(['No', 'Yes'], rotation=0)
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(housing_subscription):
    ax1.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Personal loan
ax2 = axes[1]
loan_subscription = df.groupby('loan')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_loan = ['#2ecc71', '#e67e22']
loan_subscription.plot(kind='bar', ax=ax2, color=colors_loan, edgecolor='black')
ax2.set_xlabel('Has Personal Loan?', fontsize=12, fontweight='bold')
ax2.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax2.set_title('Subscription Rate by Personal Loan Status', fontsize=13, fontweight='bold')
ax2.set_xticklabels(['No', 'Yes'], rotation=0)
ax2.grid(axis='y', alpha=0.3)
for i, v in enumerate(loan_subscription):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Credit default
ax3 = axes[2]
default_subscription = df.groupby('default')['y'].apply(lambda x: (x=='yes').sum() / len(x) * 100)
colors_default = ['#95e1d3', '#f38181']
default_subscription.plot(kind='bar', ax=ax3, color=colors_default, edgecolor='black')
ax3.set_xlabel('Has Credit Default?', fontsize=12, fontweight='bold')
ax3.set_ylabel('Subscription Rate (%)', fontsize=12, fontweight='bold')
ax3.set_title('Subscription Rate by Credit Default Status', fontsize=13, fontweight='bold')
ax3.set_xticklabels(['No', 'Yes'], rotation=0)
ax3.grid(axis='y', alpha=0.3)
for i, v in enumerate(default_subscription):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n🏦 FINANCIAL PRODUCTS INSIGHTS:")
print(f"  • Housing loan: Customers WITHOUT housing loan more likely to subscribe ({housing_subscription.loc['no']:.1f}% vs {housing_subscription.loc['yes']:.1f}%)")
print(f"  • Personal loan: Customers WITHOUT personal loan more likely to subscribe ({loan_subscription.loc['no']:.1f}% vs {loan_subscription.loc['yes']:.1f}%)")
print(f"  • Credit default: Very few defaults in dataset ({(df['default']=='yes').sum()} cases)")

print("\n" + "="*70)
print("🎯 TOP PREDICTIVE FACTORS SUMMARY")
print("="*70)
print("\nBased on subscription rate analysis:")
print(f"\n1. CONTACT METHOD is critical:")
print(f"   • {contact_subscription.idxmax()}: {contact_subscription.max():.1f}% subscription rate")
print(f"   • {contact_subscription.idxmin()}: {contact_subscription.min():.1f}% subscription rate")
print(f"   • Impact: {contact_subscription.max() - contact_subscription.min():.1f} percentage points difference")

print(f"\n2. PREVIOUS CAMPAIGN OUTCOME matters:")
print(f"   • Success: {poutcome_subscription.loc['success']:.1f}% subscription rate")
print(f"   • Failure: {poutcome_subscription.loc['failure']:.1f}% subscription rate")
print(f"   • Impact: {poutcome_subscription.loc['success'] - poutcome_subscription.loc['failure']:.1f} percentage points difference")

print(f"\n3. ACCOUNT BALANCE - Higher is better:")
print(f"   • Median balance (subscribers): ${df[df['y']=='yes']['balance'].median():,.0f}")
print(f"   • Median balance (non-subscribers): ${df[df['y']=='no']['balance'].median():,.0f}")

print(f"\n4. AGE GROUP - Sweet spot exists:")
print(f"   • Best age group: {subscription_by_age.idxmax()} ({subscription_by_age.max():.1f}% subscription rate)")

print(f"\n5. CAMPAIGN INTENSITY - Less is more:")
print(f"   • 1 contact: {campaign_subscription.iloc[0]:.1f}% subscription rate")
print(f"   • 6+ contacts: {campaign_subscription.iloc[-1]:.1f}% subscription rate")

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE - All visualizations generated!")
print("="*70)